# YOLOv8 Fine-tuning on Google Colab
Google Drive 데이터를 이용한 YOLOv8 파인튜닝

**데이터 구조 가정:**
```
/content/HTP/HTP/
  train/원천데이터/   ← 학습 이미지
  val/원천데이터/     ← 검증 이미지 (라벨 없으면 train에서 분할)
  yolo_labels/train/  ← YOLO 형식 라벨 (.txt)
```

## 1. Google Drive 마운트 및 패키지 설치

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q

## 2. 데이터 구조 탐색

In [ ]:
import os
from pathlib import Path

BASE = Path('/content/HTP/HTP')

LABEL_TRAIN_DIR = BASE / 'yolo_labels' / 'train'
IMG_TRAIN_DIR   = BASE / 'train' / '원천데이터'
IMG_VAL_DIR     = BASE / 'val'   / '원천데이터'

label_files = sorted(LABEL_TRAIN_DIR.glob('*.txt'))
img_exts    = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
train_imgs  = sorted([p for p in IMG_TRAIN_DIR.rglob('*') if p.suffix.lower() in img_exts])
val_imgs    = sorted([p for p in IMG_VAL_DIR.rglob('*')   if p.suffix.lower() in img_exts]) if IMG_VAL_DIR.exists() else []

print(f'학습 라벨:  {len(label_files)} 개  ({LABEL_TRAIN_DIR})')
print(f'학습 이미지: {len(train_imgs)} 개  ({IMG_TRAIN_DIR})')
print(f'검증 이미지: {len(val_imgs)} 개  ({IMG_VAL_DIR})')

# 클래스 ID 수집
class_ids = set()
for lf in label_files:
    for line in lf.read_text().strip().splitlines():
        if line.strip():
            class_ids.add(int(line.split()[0]))

print(f'\n감지된 클래스 ID: {sorted(class_ids)}')
print('※ 아래 셀에서 클래스 이름을 직접 입력하세요.')

## 3. 클래스 이름 설정 (필수 수정)

In [ ]:
# ↓↓↓ 클래스 ID 순서에 맞게 이름을 수정하세요 ↓↓↓
CLASS_NAMES = [
    'house',   # ID 0
    'tree',    # ID 1
    'person',  # ID 2
]
# ↑↑↑ 실제 클래스에 맞게 수정 ↑↑↑

print(f'클래스 수: {len(CLASS_NAMES)}')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {i}: {name}')

## 4. YOLO 데이터셋 구조 생성 및 train/val 분할

In [ ]:
import shutil
import random

YOLO_ROOT  = Path('/content/yolo_dataset')
VAL_RATIO  = 0.2   # 검증 데이터 비율
RANDOM_SEED = 42

for split in ['train', 'val']:
    (YOLO_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)

# 라벨이 있는 이미지만 매칭
label_stems = {lf.stem: lf for lf in label_files}
img_map     = {img.stem: img for img in train_imgs}

matched = [(label_stems[s], img_map[s]) for s in label_stems if s in img_map]
print(f'라벨-이미지 매칭: {len(matched)} / {len(label_files)} 개')
if len(matched) < len(label_files):
    unmatched = [s for s in label_stems if s not in img_map]
    print(f'  매칭 실패 라벨: {unmatched[:5]} ...')

random.seed(RANDOM_SEED)
random.shuffle(matched)
val_cut  = int(len(matched) * VAL_RATIO)
val_set  = matched[:val_cut]
train_set = matched[val_cut:]

def copy_pair(pairs, split):
    for lf, img in pairs:
        shutil.copy2(img, YOLO_ROOT / 'images' / split / img.name)
        shutil.copy2(lf,  YOLO_ROOT / 'labels' / split / lf.name)

copy_pair(train_set, 'train')
copy_pair(val_set,   'val')

print(f'\n학습 세트: {len(train_set)} 개')
print(f'검증 세트: {len(val_set)} 개')
print(f'데이터셋 위치: {YOLO_ROOT}')

## 5. data.yaml 생성

In [ ]:
import yaml

data_yaml = {
    'path'  : str(YOLO_ROOT),
    'train' : 'images/train',
    'val'   : 'images/val',
    'nc'    : len(CLASS_NAMES),
    'names' : {i: name for i, name in enumerate(CLASS_NAMES)},
}

yaml_path = YOLO_ROOT / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, allow_unicode=True, default_flow_style=False)

print('data.yaml 생성 완료:')
print(yaml_path.read_text())

## 6. 라벨 파일 이상 여부 확인

In [ ]:
errors = []
for lf in (YOLO_ROOT / 'labels' / 'train').glob('*.txt'):
    for i, line in enumerate(lf.read_text().strip().splitlines()):
        parts = line.strip().split()
        if len(parts) != 5:
            errors.append(f'{lf.name} line {i+1}: 컬럼 수 이상 ({len(parts)})')
            continue
        cid = int(parts[0])
        coords = list(map(float, parts[1:]))
        if cid >= len(CLASS_NAMES):
            errors.append(f'{lf.name} line {i+1}: 클래스 ID {cid} >= nc({len(CLASS_NAMES)})')
        if any(not (0 <= v <= 1) for v in coords):
            errors.append(f'{lf.name} line {i+1}: 좌표 범위 초과 {coords}')

if errors:
    print(f'이상 라벨 {len(errors)}건:')
    for e in errors[:10]:
        print(' ', e)
else:
    print('라벨 검증 통과 — 이상 없음')

## 7. YOLOv8 파인튜닝 실행

In [ ]:
from ultralytics import YOLO

# 모델 크기 선택: yolov8n(빠름) / yolov8s / yolov8m / yolov8l / yolov8x(정확)
MODEL     = 'yolov8n.pt'
EPOCHS    = 50
IMGSZ     = 640
BATCH     = 16   # Colab GPU 메모리 부족 시 8로 줄이기
PATIENCE  = 20   # 조기 종료

model = YOLO(MODEL)
results = model.train(
    data     = str(yaml_path),
    epochs   = EPOCHS,
    imgsz    = IMGSZ,
    batch    = BATCH,
    patience = PATIENCE,
    device   = 0,       # GPU 사용 (CPU면 'cpu')
    project  = '/content/runs/train',
    name     = 'htp_finetune',
    exist_ok = True,
    plots    = True,
)

print('\n학습 완료!')
print(f'최적 가중치: {results.save_dir}/weights/best.pt')

## 8. 학습 결과 확인

In [ ]:
from ultralytics import YOLO
from pathlib import Path

best_pt = Path('/content/runs/train/htp_finetune/weights/best.pt')
model   = YOLO(str(best_pt))

metrics = model.val(data=str(yaml_path), split='val')
print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

## 9. 가중치를 Google Drive에 저장

In [ ]:
import shutil

SAVE_DIR = Path('/content/drive/MyDrive/SAI/models')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(best_pt, SAVE_DIR / 'htp_best.pt')
print(f'저장 완료: {SAVE_DIR / "htp_best.pt"}')